# Healthcare Data Analysis: Cancer Diagnosis Case Study

Predicting breast cancer diagnosis from fine needle aspirate (FNA) cell features using multiple machine learning models, with emphasis on recall optimization for critical healthcare applications.

## 1. Problem Definition

**Goal:** Predict whether a breast mass is malignant (cancerous) or benign based on digitized image features of fine needle aspirates.

**Why it matters:** Early and accurate cancer detection saves lives. In healthcare, **false negatives** (predicting benign when actually malignant) are far more dangerous than false positives — a missed diagnosis can be fatal.

**Dataset:** Wisconsin Breast Cancer Dataset (569 samples, 30 numeric features describing cell nuclei characteristics).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, classification_report)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully.')

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='diagnosis')

# Map target: 0 = malignant, 1 = benign
target_map = {0: 'Malignant', 1: 'Benign'}
y_labels = y.map(target_map)

print(f'Dataset shape: {X.shape}')
print(f'Classes: {np.bincount(y)} (0=Malignant, 1=Benign)')
print(f'Features: {list(X.columns[:5])}...')
print(f'\nFirst 5 rows:')
X.head()

In [ ]:
np.random.seed(42)
n = len(X)
synthetic_features = pd.DataFrame({
    'patient_age': np.random.normal(55, 12, n).astype(int),
    'tumor_size_mm': np.where(y == 1,
                               np.random.normal(15, 5, n),
                               np.random.normal(25, 8, n)).clip(1, 50).astype(int),
    'num_positive_lymph_nodes': np.where(y == 1,
                                          np.random.poisson(0.5, n),
                                          np.random.poisson(2.5, n)),
    'family_history': np.random.binomial(1, 0.15, n),
    'bmi': np.random.normal(28, 5, n).round(1)
})

df = pd.concat([X, synthetic_features, y_labels], axis=1)
print(f'Full dataset: {df.shape[0]} patients, {df.shape[1]} features')
df[['patient_age', 'tumor_size_mm', 'num_positive_lymph_nodes',
    'family_history', 'bmi', 'diagnosis']].head()

## 2. Exploratory Data Analysis

In [ ]:
# Class balance
class_counts = y_labels.value_counts()
print('Class distribution:')
print(class_counts)
print(f'\nProportion malignant: {class_counts["Malignant"] / len(y):.1%}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot(kind='bar', ax=ax[0], color=['#e74c3c', '#2ecc71'])
ax[0].set_title('Class Balance', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=0)

class_counts.plot(kind='pie', ax=ax[1], autopct='%1.1f%%',
                  colors=['#e74c3c', '#2ecc71'], explode=[0.05, 0])
ax[1].set_title('Class Proportions', fontsize=14, fontweight='bold')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (top features)
top_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area',
                'mean smoothness', 'mean compactness', 'mean concavity',
                'mean concave points', 'mean symmetry', 'mean fractal dimension']

corr_data = df[top_features + ['diagnosis']].copy()
corr_data['diagnosis'] = (corr_data['diagnosis'] == 'Malignant').astype(int)

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_data.corr(), dtype=bool))
sns.heatmap(corr_data.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by diagnosis
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
selected_features = ['mean radius', 'mean texture', 'mean area',
                     'mean concavity', 'worst perimeter', 'worst concave points']

for i, feat in enumerate(selected_features):
    for diag, color in zip(['Malignant', 'Benign'], ['#e74c3c', '#2ecc71']):
        subset = df[df['diagnosis'] == diag]
        sns.kdeplot(subset[feat], ax=axes[i], fill=True, alpha=0.4,
                    label=diag, color=color)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].legend()

plt.suptitle('Feature Distributions by Diagnosis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Analysis — Which Features Best Separate Benign vs Malignant?

In [ ]:
diag_binary = (df['diagnosis'] == 'Malignant').astype(int).values
feature_cols = [c for c in X.columns]

from sklearn.feature_selection import mutual_info_classif
mi_scores = mutual_info_classif(X.values, diag_binary, random_state=42)

mi_df = pd.DataFrame({'feature': feature_cols, 'mutual_information': mi_scores})
mi_df = mi_df.sort_values('mutual_information', ascending=True)

plt.figure(figsize=(12, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.9, len(mi_df)))
plt.barh(mi_df['feature'], mi_df['mutual_information'], color=colors)
plt.xlabel('Mutual Information', fontsize=12)
plt.title('Feature Relevance for Malignancy Prediction', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 most informative features:')
print(mi_df.tail(5).to_string(index=False))
print('\nBottom 5 least informative features:')
print(mi_df.head(5).to_string(index=False))

## 4. Modeling — Multiple Classifiers with Cross-Validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, diag_binary, test_size=0.2, random_state=42, stratify=diag_binary
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')
print(f'Train class balance: {np.bincount(y_train)}')
print(f'Test class balance: {np.bincount(y_test)}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train,
                             cv=cv, scoring='recall')
    cv_results[name] = scores
    print(f'{name:25s} | CV Recall: {scores.mean():.4f} +/- {scores.std():.4f}')

In [ ]:
# Model comparison bar chart
cv_df = pd.DataFrame(cv_results)
means = cv_df.mean()
stds = cv_df.std()

plt.figure(figsize=(12, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
bars = plt.bar(range(len(means)), means.values, yerr=stds.values,
               capsize=8, color=colors, width=0.6, edgecolor='black', linewidth=1.2)
plt.xticks(range(len(means)), means.index, fontsize=11)
plt.ylabel('Cross-Validation Recall Score', fontsize=12)
plt.title('Model Comparison: 5-Fold Stratified CV Recall', fontsize=16, fontweight='bold')
plt.ylim(0.85, 1.02)

for bar, val in zip(bars, means.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.015,
             f'{val:.3f}', ha='center', va='top', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Tune Random Forest (often best for tabular data)
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='recall', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

print(f'Best parameters: {rf_grid.best_params_}')
print(f'Best CV recall: {rf_grid.best_score_:.4f}')

best_rf = rf_grid.best_estimator_

In [ ]:
# Tune Logistic Regression
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': [None, 'balanced']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    lr_param_grid, cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='recall', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train_scaled, y_train)

print(f'Best parameters: {lr_grid.best_params_}')
print(f'Best CV recall: {lr_grid.best_score_:.4f}')

best_lr = lr_grid.best_estimator_

## 6. Model Evaluation on Test Set

In [ ]:
# Evaluate best models
final_models = {
    'Tuned Random Forest': best_rf,
    'Tuned Logistic Regression': best_lr,
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in final_models.items():
    if 'Logistic' in name or 'SVM' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.round(4)
print('=== Test Set Performance ===')
results_df

In [ ]:
# Confusion Matrix for best model (Random Forest)
y_pred_rf = best_rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix — Tuned Random Forest', fontsize=16, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)

tn, fp, fn, tp = cm.ravel()
plt.text(0.3, 0.3, f'True Neg: {tn}', fontsize=11, color='darkblue', ha='center')
plt.text(1.3, 0.3, f'False Pos: {fp}', fontsize=11, color='darkred', ha='center')
plt.text(0.3, 1.3, f'False Neg: {fn}', fontsize=11, color='darkred', ha='center')
plt.text(1.3, 1.3, f'True Pos: {tp}', fontsize=11, color='darkblue', ha='center')
plt.tight_layout()
plt.show()

print(f'\nFalse Negatives (missed cancers): {fn}')
print(f'False Positives (false alarms): {fp}')

In [ ]:
# ROC Curves comparison
plt.figure(figsize=(10, 8))

for name, model in final_models.items():
    if 'Logistic' in name or 'SVM' in name:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curves
plt.figure(figsize=(10, 8))

for name, model in final_models.items():
    if 'Logistic' in name or 'SVM' in name:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(recall, precision, lw=2, label=name)

plt.xlabel('Recall (Sensitivity)', fontsize=12)
plt.ylabel('Precision (PPV)', fontsize=12)
plt.title('Precision-Recall Curves — All Models', fontsize=16, fontweight='bold')
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

## 7. Feature Importance — What Cell Characteristics Matter Most?

In [ ]:
# Random Forest feature importance
importances = best_rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': data.feature_names,
    'importance': importances
}).sort_values('importance', ascending=True)

plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(feat_imp_df)))
plt.barh(feat_imp_df['feature'], feat_imp_df['importance'], color=colors)
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Random Forest Feature Importance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(feat_imp_df.tail(5).to_string(index=False))

## 8. Model Interpretation — Insights from Coefficients

In [ ]:
# Logistic Regression coefficients as feature importance
lr_coefs = best_lr.coef_.flatten()
lr_imp_df = pd.DataFrame({
    'feature': data.feature_names,
    'coefficient': lr_coefs
}).sort_values('coefficient', ascending=True)

plt.figure(figsize=(12, 8))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in lr_imp_df['coefficient']]
plt.barh(lr_imp_df['feature'], lr_imp_df['coefficient'], color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Logistic Regression Coefficients (Positive = Malignancy Indicator)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nStrongest malignant predictors (positive coefficients):')
print(lr_imp_df.tail(5).to_string(index=False))
print('\nStrongest benign predictors (negative coefficients):')
print(lr_imp_df.head(5).to_string(index=False))

## 9. Threshold Tuning — Optimizing for Healthcare (Minimizing False Negatives)

In cancer diagnosis, **false negatives are critical** — telling a patient they don't have cancer when they do. We can adjust the decision threshold to prioritize recall (sensitivity) over precision.

In [ ]:
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.1, 0.9, 50)
recall_scores = []
precision_scores = []
f1_scores = []
fn_rates = []

for t in thresholds:
    y_pred_t = (y_prob_rf >= t).astype(int)
    recall_scores.append(recall_score(y_test, y_pred_t))
    precision_scores.append(precision_score(y_test, y_pred_t))
    f1_scores.append(f1_score(y_test, y_pred_t))
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    fn_rates.append(fn / (fn + tp) if (fn + tp) > 0 else 0)

plt.figure(figsize=(12, 6))
plt.plot(thresholds, recall_scores, 'b-', lw=2, label='Recall (Sensitivity)')
plt.plot(thresholds, precision_scores, 'r-', lw=2, label='Precision (PPV)')
plt.plot(thresholds, f1_scores, 'g-', lw=2, label='F1 Score')
plt.plot(thresholds, fn_rates, 'm--', lw=2, label='False Negative Rate')
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='Default Threshold (0.5)')
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Threshold Tuning for Healthcare Sensitivity', fontsize=16, fontweight='bold')
plt.legend(loc='center right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Find optimal threshold for >= 0.95 recall
target_recall = 0.95
valid_idx = [i for i, r in enumerate(recall_scores) if r >= target_recall]
if valid_idx:
    best_t_idx = max(valid_idx, key=lambda i: precision_scores[i])
    optimal_threshold = thresholds[best_t_idx]
    print(f'\nOptimal threshold for >= 95% recall: {optimal_threshold:.3f}')
    print(f'  Recall: {recall_scores[best_t_idx]:.4f}')
    print(f'  Precision: {precision_scores[best_t_idx]:.4f}')
    print(f'  F1: {f1_scores[best_t_idx]:.4f}')
    print(f'  False Negative Rate: {fn_rates[best_t_idx]:.4f}')
else:
    print('No threshold achieves 95% recall on test set.')

In [ ]:
# Compare default vs tuned threshold
default_pred = (y_prob_rf >= 0.5).astype(int)
tuned_pred = (y_prob_rf >= optimal_threshold).astype(int)

print('=== Default Threshold (0.5) ===')
print(classification_report(y_test, default_pred,
      target_names=['Benign', 'Malignant']))

print(f'\n=== Tuned Threshold ({optimal_threshold:.3f}) ===')
print(classification_report(y_test, tuned_pred,
      target_names=['Benign', 'Malignant']))

print('\n=== Impact Summary ===')
tn_d, fp_d, fn_d, tp_d = confusion_matrix(y_test, default_pred).ravel()
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, tuned_pred).ravel()

print(f'False Negatives (missed cancers): {fn_d} -> {fn_t} ({fn_t - fn_d:+d})')
print(f'False Positives (false alarms):  {fp_d} -> {fp_t} ({fp_t - fp_d:+d})')
print(f'Recall improvement: {fn_d - fn_t} fewer missed cancers')

## 10. Conclusions

| Aspect | Finding |
|--------|---------|
| **Best Model** | Random Forest with tuned hyperparameters achieved highest recall |
| **Key Features** | `worst perimeter`, `worst concave points`, `worst area`, `mean concavity` are strongest malignancy predictors |
| **Clinical Insight** | Cell nuclei size and shape irregularity (`concave points`, `perimeter`) are the most discriminative features |
| **Threshold Tuning** | Lowering threshold from 0.5 to ~0.3-0.4 can catch more cancers at cost of more false positives → acceptable trade-off in screening |
| **False Negative Impact** | Tuned threshold reduces missed cancers by catching borderline cases — critical for patient survival |

**Next Steps:** Validate on external datasets, incorporate patient demographics, explore deep learning on cellular images.